# Voyage Multimodal Image Embeddings — PCA + K-Means Demo POC

This notebook is designed to be demo-safe:

- Uses **real curated Caltech-101 images**, not synthetic images and not random web images.
- Uses the **current CaltechDATA zip download URL** instead of the older broken `101_ObjectCategories.tar.gz` URL.
- Reads `VOYAGE_API_KEY` from your existing `.env` file.
- Keeps API usage low for the Voyage free tier.
- Caches embeddings locally so reruns do not repeatedly call the API.
- Uses PCA for visualization and denoising, then K-Means for grouping similar images.

> First run with the defaults. After the cache is created, you can increase `N_PER_CLASS` slowly.


In [ ]:
# If any package is missing, uncomment and run this once.
# %pip install -q voyageai python-dotenv requests pillow pandas numpy scikit-learn matplotlib


In [ ]:
from pathlib import Path
import os
import time
import hashlib
import tarfile
import zipfile
import shutil
from typing import List, Optional

import numpy as np
import pandas as pd
import requests
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

# -------------------------
# Free-tier friendly config
# -------------------------
RANDOM_SEED = 42
MODEL = "voyage-multimodal-3.5"

BASE_DIR = Path("voyage_caltech101_pca_kmeans_demo")
RAW_DIR = BASE_DIR / "raw_data"
SELECTED_DIR = BASE_DIR / "selected_images"
CACHE_DIR = BASE_DIR / "cache"

for p in [RAW_DIR, SELECTED_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Keep this small for Voyage free tier.
# Total API embeddings = len(SELECTED_CLASSES) * N_PER_CLASS.
SELECTED_CLASSES = [
    "airplanes",
    "Motorbikes",
    "pizza",
    "sunflower",
    "dalmatian",
    "laptop",
]
N_PER_CLASS = 4
MAX_IMAGE_SIDE = 512

# Conservative free-tier settings.
BATCH_SIZE = 2
DELAY_SECONDS_BETWEEN_BATCHES = 5
MAX_RETRIES = 4

# Current official CaltechDATA file URL.
CALTECH101_ZIP_URL = "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1"
CALTECH101_ZIP_PATH = RAW_DIR / "caltech-101.zip"

print("Expected total images:", len(SELECTED_CLASSES) * N_PER_CLASS)
print("Base directory:", BASE_DIR.resolve())


In [ ]:
def find_dotenv_file(start: Optional[Path] = None) -> Optional[Path]:
    """Find .env in the current directory or any parent directory."""
    start = Path.cwd() if start is None else Path(start)
    for folder in [start] + list(start.parents):
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    return None


def load_env_key(key: str = "VOYAGEAI_API_KEY") -> str:
    """Load a key from environment or .env without requiring python-dotenv."""
    value = os.getenv(key)
    if value:
        return value.strip().strip('"').strip("'")

    # Try python-dotenv if available.
    try:
        from dotenv import load_dotenv
        dotenv_path = find_dotenv_file()
        if dotenv_path:
            load_dotenv(dotenv_path)
            value = os.getenv(key)
            if value:
                print(f"Loaded {key} from {dotenv_path}")
                return value.strip().strip('"').strip("'")
    except Exception:
        pass

    # Manual .env parse fallback.
    dotenv_path = find_dotenv_file()
    if dotenv_path:
        with open(dotenv_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                if k.strip() == key:
                    print(f"Loaded {key} from {dotenv_path}")
                    return v.strip().strip('"').strip("'")

    raise RuntimeError(
        f"Could not find {key}. Please make sure your .env file contains a line like:\n"
        f"{key}=your_api_key_here"
    )

VOYAGEAI_API_KEY = load_env_key("VOYAGEAI_API_KEY")
print("VOYAGEAI_API_KEY found in environment/.env")


In [ ]:
def download_file_with_checks(url: str, destination: Path, min_bytes: int = 10_000_000) -> Path:
    """Download a file with headers, retries, and size checks."""
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size >= min_bytes:
        print(f"Using existing file: {destination} ({destination.stat().st_size / 1_000_000:.1f} MB)")
        return destination

    headers = {
        "User-Agent": "Mozilla/5.0 image-embedding-demo/1.0",
        "Accept": "application/zip,application/octet-stream,*/*",
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"Downloading Caltech-101 archive, attempt {attempt}/{MAX_RETRIES}...")
            with requests.get(url, headers=headers, stream=True, timeout=90, allow_redirects=True) as response:
                response.raise_for_status()
                tmp_path = destination.with_suffix(destination.suffix + ".partial")
                with open(tmp_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                tmp_path.replace(destination)

            size = destination.stat().st_size
            if size < min_bytes:
                raise RuntimeError(f"Downloaded file is too small: {size} bytes")

            print(f"Downloaded: {destination} ({size / 1_000_000:.1f} MB)")
            return destination

        except Exception as exc:
            last_error = exc
            print(f"Download attempt {attempt} failed: {repr(exc)}")
            time.sleep(min(5 * attempt, 20))

    raise RuntimeError(
        "Could not download Caltech-101 from the official CaltechDATA URL.\n"
        f"URL: {url}\n"
        f"Last error: {repr(last_error)}\n\n"
        "Manual fallback: download caltech-101.zip from the CaltechDATA page and place it here:\n"
        f"{destination.resolve()}"
    )


def safe_extract_tar(tar_path: Path, extract_to: Path) -> None:
    """Safely extract tar archives without path traversal risk."""
    extract_to.mkdir(parents=True, exist_ok=True)
    base = extract_to.resolve()
    with tarfile.open(tar_path, "r:*") as tar:
        for member in tar.getmembers():
            member_path = (extract_to / member.name).resolve()
            if not str(member_path).startswith(str(base)):
                raise RuntimeError(f"Unsafe path in tar archive: {member.name}")
        tar.extractall(extract_to)


def find_categories_dir(root: Path, required_classes: List[str]) -> Optional[Path]:
    """Find a folder that contains the selected Caltech class folders."""
    candidates = [p for p in root.rglob("101_ObjectCategories") if p.is_dir()]
    candidates += [p for p in root.rglob("*") if p.is_dir() and any((p / c).is_dir() for c in required_classes)]

    for candidate in candidates:
        hits = sum((candidate / c).is_dir() for c in required_classes)
        if hits >= max(2, len(required_classes) // 2):
            return candidate
    return None


def prepare_caltech101() -> Path:
    """Download and extract Caltech-101 using the current official zip package."""
    existing = find_categories_dir(RAW_DIR, SELECTED_CLASSES)
    if existing:
        print(f"Using existing extracted Caltech-101 folder: {existing}")
        return existing

    zip_path = download_file_with_checks(CALTECH101_ZIP_URL, CALTECH101_ZIP_PATH)

    # Extract the outer zip.
    print("Extracting outer caltech-101.zip...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(RAW_DIR / "caltech101_zip_extracted")

    existing = find_categories_dir(RAW_DIR, SELECTED_CLASSES)
    if existing:
        print(f"Found extracted categories folder: {existing}")
        return existing

    # The zip normally contains 101_ObjectCategories.tar.gz. Locate and extract it.
    tar_candidates = list(RAW_DIR.rglob("101_ObjectCategories.tar.gz"))
    if not tar_candidates:
        raise RuntimeError(
            "Could not find 101_ObjectCategories.tar.gz inside caltech-101.zip. "
            "Please inspect the downloaded zip structure."
        )

    object_tar = tar_candidates[0]
    print(f"Extracting inner archive: {object_tar}")
    safe_extract_tar(object_tar, RAW_DIR / "caltech101_objects_extracted")

    existing = find_categories_dir(RAW_DIR, SELECTED_CLASSES)
    if not existing:
        raise RuntimeError("Could not locate extracted 101_ObjectCategories folder after extraction.")

    print(f"Ready: {existing}")
    return existing

categories_dir = prepare_caltech101()
print("Categories directory:", categories_dir)
print("Available selected folders:", [c for c in SELECTED_CLASSES if (categories_dir / c).is_dir()])


In [ ]:
def resize_and_copy_image(src_path: Path, dst_path: Path, max_side: int = 512) -> bool:
    """Open, validate, resize, and save one image. Returns False if the image is unreadable."""
    try:
        img = Image.open(src_path).convert("RGB")
        img.thumbnail((max_side, max_side))
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        img.save(dst_path, quality=95)
        return True
    except Exception as exc:
        print(f"Skipping unreadable image: {src_path} -> {repr(exc)}")
        return False


def build_demo_metadata(categories_dir: Path) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED)
    rows = []

    # Clean selected dir to avoid mixing old and new selections.
    if SELECTED_DIR.exists():
        shutil.rmtree(SELECTED_DIR)
    SELECTED_DIR.mkdir(parents=True, exist_ok=True)

    for label_id, class_name in enumerate(SELECTED_CLASSES):
        class_dir = categories_dir / class_name
        if not class_dir.exists():
            print(f"Missing class folder, skipping: {class_name}")
            continue

        images = []
        for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
            images.extend(class_dir.glob(ext))
        images = sorted(images)

        if not images:
            print(f"No image files found for class: {class_name}")
            continue

        selected_count = min(N_PER_CLASS, len(images))
        selected_idx = rng.choice(len(images), size=selected_count, replace=False)
        selected_images = [images[i] for i in selected_idx]

        safe_label = class_name.replace("/", "_").replace(" ", "_")
        saved = 0
        for src in selected_images:
            dst = SELECTED_DIR / safe_label / f"{safe_label}_{saved:03d}.jpg"
            if resize_and_copy_image(src, dst, MAX_IMAGE_SIDE):
                rows.append({
                    "label_id": label_id,
                    "label": class_name,
                    "source_path": str(src),
                    "image_path": str(dst),
                })
                saved += 1

    meta = pd.DataFrame(rows)
    if meta.empty:
        raise RuntimeError("No images were selected. Check the Caltech extraction and class names.")

    meta = meta.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Images selected: {len(meta)}")
    print(f"Classes selected: {meta['label'].nunique()}")
    display(meta.head())
    display(meta["label"].value_counts().sort_index())
    return meta

meta = build_demo_metadata(categories_dir)


In [ ]:
def show_image_grid(df: pd.DataFrame, n: int = 24, cols: int = 6, title_col: str = "label") -> None:
    subset = df.head(min(n, len(df))).copy()
    rows = int(np.ceil(len(subset) / cols))
    plt.figure(figsize=(cols * 2.2, rows * 2.5))

    for i, row in enumerate(subset.itertuples(index=False), start=1):
        img = Image.open(row.image_path).convert("RGB")
        ax = plt.subplot(rows, cols, i)
        ax.imshow(img)
        ax.set_title(getattr(row, title_col), fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_image_grid(meta, n=len(meta), cols=6)


In [ ]:
def fingerprint_image_selection(meta: pd.DataFrame) -> str:
    parts = []
    for p in meta["image_path"].tolist():
        path = Path(p)
        stat = path.stat()
        parts.append(f"{path.as_posix()}|{stat.st_size}|{int(stat.st_mtime)}")
    text = "\n".join(parts) + f"\nMODEL={MODEL}"
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:16]


def embed_images_with_voyage(meta: pd.DataFrame) -> np.ndarray:
    import voyageai

    fp = fingerprint_image_selection(meta)
    cache_path = CACHE_DIR / f"voyage_image_embeddings_{MODEL}_{fp}.npy"
    meta_cache_path = CACHE_DIR / f"voyage_image_embeddings_{MODEL}_{fp}_meta.csv"

    if cache_path.exists():
        print(f"Loading cached embeddings: {cache_path}")
        return np.load(cache_path)

    client = voyageai.Client(api_key=VOYAGEAI_API_KEY)
    all_embeddings = []
    paths = meta["image_path"].tolist()

    print(f"Embedding {len(paths)} images using {MODEL}...")
    print("This cell calls the Voyage API. Keep defaults small for free tier.")

    for start in range(0, len(paths), BATCH_SIZE):
        batch_paths = paths[start:start + BATCH_SIZE]
        batch_inputs = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            batch_inputs.append([img])  # image only; labels are NOT sent to Voyage

        last_error = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                result = client.multimodal_embed(
                    inputs=batch_inputs,
                    model=MODEL,
                    input_type="document",
                )
                all_embeddings.extend(result.embeddings)
                print(f"Embedded {min(start + len(batch_paths), len(paths))}/{len(paths)}")
                break
            except Exception as exc:
                last_error = exc
                wait = min(20 * attempt, 90)
                print(f"Voyage API attempt {attempt} failed: {repr(exc)}")
                print(f"Waiting {wait} seconds before retry...")
                time.sleep(wait)
        else:
            raise RuntimeError(f"Voyage embedding failed after retries. Last error: {repr(last_error)}")

        if start + BATCH_SIZE < len(paths):
            time.sleep(DELAY_SECONDS_BETWEEN_BATCHES)

    embeddings = np.array(all_embeddings, dtype=np.float32)
    np.save(cache_path, embeddings)
    meta.to_csv(meta_cache_path, index=False)
    print(f"Saved embeddings cache: {cache_path}")
    return embeddings

embeddings = embed_images_with_voyage(meta)
print("Embeddings shape:", embeddings.shape)


In [ ]:
# Normalize embeddings before cosine-style similarity, PCA, and K-Means.
X = normalize(embeddings)

# 2D PCA for visualization.
pca_2d = PCA(n_components=2, random_state=RANDOM_SEED)
X_2d = pca_2d.fit_transform(X)

# PCA for denoising before K-Means.
n_components_cluster = min(50, X.shape[0] - 1, X.shape[1])
pca_cluster = PCA(n_components=n_components_cluster, random_state=RANDOM_SEED)
X_pca_cluster = pca_cluster.fit_transform(X)
X_pca_cluster = normalize(X_pca_cluster)

k = meta["label"].nunique()
kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=20)
cluster_id = kmeans.fit_predict(X_pca_cluster)

results = meta.copy()
results["cluster_id"] = cluster_id
results["pca_x"] = X_2d[:, 0]
results["pca_y"] = X_2d[:, 1]

print("PCA explained variance, first 2 components:", pca_2d.explained_variance_ratio_)
print("PCA components used before K-Means:", n_components_cluster)
display(results.head())


In [ ]:
def cluster_purity(true_labels, predicted_clusters) -> float:
    df = pd.DataFrame({"true": true_labels, "cluster": predicted_clusters})
    total_correct = 0
    for _, group in df.groupby("cluster"):
        total_correct += group["true"].value_counts().iloc[0]
    return total_correct / len(df)

purity = cluster_purity(results["label"], results["cluster_id"])
ari = adjusted_rand_score(results["label"], results["cluster_id"])
nmi = normalized_mutual_info_score(results["label"], results["cluster_id"])

silhouette = None
if 1 < k < len(results):
    silhouette = silhouette_score(X_pca_cluster, results["cluster_id"])

metrics = pd.DataFrame([
    {"metric": "cluster_purity", "value": purity},
    {"metric": "adjusted_rand_index", "value": ari},
    {"metric": "normalized_mutual_info", "value": nmi},
    {"metric": "silhouette_score", "value": silhouette},
])

display(metrics)

cluster_summary = (
    results.groupby("cluster_id")["label"]
    .value_counts()
    .rename("count")
    .reset_index()
    .sort_values(["cluster_id", "count"], ascending=[True, False])
)
display(cluster_summary)


In [ ]:
plt.figure(figsize=(10, 7))
for label, group in results.groupby("label"):
    plt.scatter(group["pca_x"], group["pca_y"], label=label, s=80, alpha=0.85)

for row in results.itertuples():
    plt.text(row.pca_x, row.pca_y, str(row.cluster_id), fontsize=8)

plt.title("Voyage Image Embeddings — PCA 2D View\nColor = true class, number = K-Means cluster")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
def show_cluster_images(results: pd.DataFrame, max_per_cluster: int = 8) -> None:
    for cluster in sorted(results["cluster_id"].unique()):
        group = results[results["cluster_id"] == cluster].head(max_per_cluster)
        cols = min(4, len(group))
        rows = int(np.ceil(len(group) / cols))
        plt.figure(figsize=(cols * 2.4, rows * 2.7))
        plt.suptitle(f"Cluster {cluster}", fontsize=14)

        for i, row in enumerate(group.itertuples(index=False), start=1):
            img = Image.open(row.image_path).convert("RGB")
            ax = plt.subplot(rows, cols, i)
            ax.imshow(img)
            ax.set_title(row.label, fontsize=10)
            ax.axis("off")

        plt.tight_layout()
        plt.show()

show_cluster_images(results)


## How to scale after the demo

Once the notebook runs end-to-end with the defaults:

1. Increase `N_PER_CLASS` from `4` to `8`.
2. Keep `BATCH_SIZE = 2` and caching enabled for the free tier.
3. Do not delete the `cache` folder unless you intentionally want to re-embed images.
4. Add/remove Caltech classes in `SELECTED_CLASSES`.
5. For your own business dataset, replace the Caltech loader with a local folder loader using the same `meta` schema:
   - `label`
   - `image_path`

The clustering is unsupervised: labels are only used for evaluation and plot coloring. Labels are not sent to Voyage.
